In [ ]:
import os
from dashscope import MultiModalConversation

# https://help.aliyun.com/zh/model-studio/vision?spm=5176.28630291.0.0.24bf7eb5h3MdqT&disableWebsiteRedirect=true#6b5c3f098fjfc


def video_understanding(video_path, prompt, fps = 10):
    if isinstance(video_path, str):
        print("loading a video:", video_path)
    else:
        print("loading a list of images")
 
    messages = [{'role': 'system',
                    'content': [{'text': 'You are a helpful assistant.'}]},
                    {'role':'user',
                    # fps参数控制视频抽帧数量，表示每隔1/fps 秒抽取一帧
                    'content': [{'video': video_path,"fps":fps},
                                {'text': prompt}]}]
    response = MultiModalConversation.call(
        # 若没有配置环境变量，请用百炼API Key将下行替换为：api_key="sk-xxx"
        api_key=os.getenv('DASHSCOPE_API_KEY'),
        model='qwen2.5-vl-72b-instruct',
        messages=messages)
    try:
        print(response["output"]["choices"][0]["message"].content[0]["text"])
    except:
        print(response['message'])

def save_video_frames(video_path, output_folder, target_fps=None):
    """
    Save frames from a video file as images with a specific FPS.
    
    Args:
        video_path (str): Path to the input video file
        output_folder (str): Folder to save the extracted frames
        target_fps (float): Desired frames per second (if None, saves all frames)
    """
    import cv2
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # delete all files in the folder
    for file in os.listdir(output_folder):
        file_path = os.path.join(output_folder, file)
        try:
            if os.path.isfile(file_path):
                os.remove(file_path)
        except Exception as e:
            print(f"Error deleting file {file_path}: {e}")
    
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("Error: Could not open video file")
        return
    
    # Get video properties
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / original_fps
    
    print(f"Original FPS: {original_fps}")
    print(f"Total frames: {total_frames}")
    print(f"Duration: {duration:.2f} seconds")
    
    # If target_fps is None, save all frames
    if target_fps is None:
        target_fps = original_fps
    
    # Calculate frame interval based on target FPS
    frame_interval = max(1, int(round(original_fps / target_fps)))
    print(f"Saving 1 frame every {frame_interval} frames")
    
    frame_count = 0
    saved_count = 0
    
    image_path_list = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        # Save frame if it's the right interval
        if frame_count % frame_interval == 0:
            frame_filename = os.path.join(output_folder, f"frame_{saved_count:05d}.jpg")
            cv2.imwrite(frame_filename, frame)
            image_path_list.append(frame_filename)
            saved_count += 1
            
        frame_count += 1
    
    cap.release()
    print(f"Finished processing. Saved {saved_count} frames.")
    return image_path_list



## Define the task

In [ ]:

## NOTE: you can rename the objects to help VLM understanding.
task_name = "two_arm_three_piece_assembly"
object_list = ['piece_1', 'piece_2', 'base']
# task_name = "two_arm_threading"
# object_list = ['needle_obj', 'tripod_obj']
# task_name = "two_arm_lift_tray"
# object_list = ['blue_block', 'green_block', 'tray']
# task_name = "two_arm_transport"
# object_list = ['trash', 'payload', 'transport_start_bin', 'transport_start_bin_lid', 'transport_target_bin', 'trash_target_bin']

robot_list = ['robot_left', 'robot_right']
surface_list = ['table']


local_path = f"/home/user/yzchen_ws/imitation_learning/dexmimicgen/datasets/pc_demos/{task_name}/raw/{task_name}_demo_0.mp4"
qwen_vid_path =f"file://{local_path}"

use_image_list = True
target_fps = 2  # Set to None to save all frames

if use_image_list:
    output_folder = local_path.split('.')[0]
    input_path = save_video_frames(local_path, output_folder, target_fps)
else:
    input_path = qwen_vid_path


## Write prompts

In [ ]:

prompt = f"""
Task: {task_name}
Objects in the scene: {object_list}
Robots in the scene: {robot_list}
Surfaces in the scene: {surface_list}
Answer the following questions step by step based on the given video:
    1. Use objects, robots, and surface as nodes, and the contact relationship among them as edges, construct a graph to describe the initial contact relationship among them. The <initail_graph> is made up of a list of edges and nodes of the whole scene. Initially, all objects should connect to the surface.
    2.  Starting from <initail_graph>, detect contact mode changes using video frame analysis. Primitive to add a new edge: grasp (object-gripper), attach object (object-object), place (object-surface). Primitives to remove an edge: release (robot-object), lift (object-surface), dettach object (object-object). Note: 1) do not miss the contact between object and the surface. 2) For each contact change, only one <edge> should be added or removed from the graph, with the <opeartion> be "add" or "remove". 3) No explaination needed.
        
After that, provide the final output in a json format.
Your final output should be like:
    ```json
    {{
        "list of nodes": "[<node1>, <node2>, ...]",
        "list of primitives": "[<primitive1>, <primitive2>, ...]",
        "initial_graph": "<initial_graph>",
        "ModeChangeDetection": [
            {{

                "graph_number": 1,
                "contact_change": "[<edge>, <opeartion>]",
                "description": "<primitive>",
            }},
            ...
        ]
    }}
    '''
"""